<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Evaluating K-Means Clusters with the Iris Dataset - Solution

---

Below you will practice using K-Means clustering and the various evaluation strategies we covered on the famous Iris dataset.

In [ ]:
%matplotlib inline 

import pandas as pd
import numpy as np
from sklearn import cluster, metrics
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('ggplot') 
from sklearn import datasets

### 1: Import and format the data

Both sklearn and seaborn have ways to import the iris data:
- `sklearn.datasets.load_iris()`
- `sns.load_dataset("iris")`

The seaborn way is easier.

In [ ]:
# The Hard Way
iris = datasets.load_iris()

In [ ]:
X = pd.DataFrame(iris.data)
X.columns = ['Sepal_Length','Sepal_Width','Petal_Length','Petal_Width']
y = pd.DataFrame(iris.target)
y.columns = ['Targets']

In [ ]:
# The Easy Way

In [ ]:
import seaborn as sns
sns_iris = sns.load_dataset("iris")

In [ ]:
sns_iris.head()

In [ ]:
X = sns_iris.iloc[:, :-1]
y = sns_iris.species

### 2. Plot the data to visually estimate to correct number of clusters

In [ ]:
X.plot(kind='scatter', x='sepal_length', y='petal_length')

In [ ]:
sns.pairplot(X)

I spy 2 very distinct clusters.
![2 clusters](./2 cluster.png)

I spy 4 somewhat distinct clusters.  2 which are dense and 2 which are not so dense or populous.
![4 cluster](./4 cluster.png)

A bit more insane but we could draw 5 straight (non-horizontal lines) to divide our data into 6 more specific clusters...
![5 split](./6 cluster.png)


### 3. Cluster the data using K-Means

- Select a number of clusters of your choice based on your visual analysis above.

In [ ]:
# lets say I choose K=4
k = 4
kmeans = cluster.KMeans(n_clusters=k)
kmeans.fit(X)

**3.2 Compute the labels and centroids.**

In [ ]:
labels = kmeans.labels_
centroids = kmeans.cluster_centers_

### 4. Visually evaluate the clusters.
- Compare the predicted labels vs. the actual labels.

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y)
le.classes_

In [ ]:
species_ind = le.transform(y)

In [ ]:
colormap = np.array(['red', 'blue', 'yellow', 'green'])

plt.subplot(1, 2, 1)
plt.scatter(X.petal_length, X.petal_width, c=colormap[species_ind], s=40)
plt.title('Actual Classification')
 
plt.subplot(1, 2, 2)
plt.scatter(X.petal_length, X.petal_width, c=colormap[labels], s=40)
plt.title('K-Means Classification')

### 5. Check the centroids and plot them along two of the features.

In [ ]:
print(centroids)

In [ ]:
fig, ax = plt.subplots(figsize=(7,6))
ax.scatter(X.petal_length, X.petal_width, c='grey', alpha=0.55, s=30)
ax.scatter(centroids[:,2], centroids[:,3], c='darkred', s=120)
plt.show()

### 6. Compute the silhoutte score for your clusters.

What does the score indicate?

In [ ]:
metrics.silhouette_score(X, labels, metric='euclidean')

In [ ]:
# The score is positive (though not 1.), indicating that there is decent separation
# and coherence with 4 clusters.

### 7. Plot the silhouette score for K = 2,3,4,5,6,7,8

You will need to fit a new model for each one. You can standardize your data before iterating through the clusters or not, up to you.

In [ ]:
from sklearn.preprocessing import StandardScaler

ss = StandardScaler()
Xs = ss.fit_transform(X)

sils = []
for k in [2,3,4,5,6,7,8]:
    km = cluster.KMeans(n_clusters=k)
    km.fit(Xs)
    sils.append(metrics.silhouette_score(Xs, km.labels_, metric='euclidean'))
    
fig, ax = plt.subplots(figsize=(8,5))
ax.plot([2,3,4,5,6,7,8], sils, c='grey', lw=3.5, alpha=0.7)
ax.scatter([2,3,4,5,6,7,8], sils, c='darkred', s=175)
plt.show()

### 8. Plot the inertia score for the different K clusters.

Is there an "elbow" to select a good number of clusters or not?

In [ ]:
inertias = []
for k in [2,3,4,5,6,7,8]:
    km = cluster.KMeans(n_clusters=k)
    km.fit(Xs)
    inertias.append(km.inertia_)
    
fig, ax = plt.subplots(figsize=(8,5))
ax.plot([2,3,4,5,6,7,8], inertias, c='grey', lw=3.5, alpha=0.7)
ax.scatter([2,3,4,5,6,7,8], inertias, c='steelblue', s=175)
plt.show()

### 9. Fit a K-Means with 3 clusters and pull out the cluster labels. Pull out the true labels as well.

Once you have both, adjust the predicted cluster labels to correspond to the true labels. For example, cluster 0 should correspond roughly to species 0, cluster 1 to species 1, and cluster 2 to species 2.

In [ ]:
species_ind

In [ ]:
km3 = cluster.KMeans(n_clusters=3)
km3.fit(Xs)

labels = km3.labels_
labels

In [ ]:
# 0 is obvious, 1 and 2 seem good enough 

### 10. Calculate the completeness, homogeneity, V measure, and mutual information (adjusted) using the predicted clusters and the true labels.

In [ ]:
from sklearn.metrics import completeness_score, homogeneity_score, v_measure_score, adjusted_mutual_info_score

In [ ]:
print(completeness_score(species_ind, labels))

In [ ]:
print(homogeneity_score(species_ind, labels))

In [ ]:
print(v_measure_score(species_ind, labels))

In [ ]:
print(adjusted_mutual_info_score(species_ind, labels))